In [10]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [11]:
df = pd.read_csv("trendyol.csv")
df = df[['review_body', 'review_rating']].dropna()

df = df[df['review_rating'] != 3]
df['label'] = df['review_rating'].apply(lambda x: 1 if x > 3 else 0)

X = df['review_body'].values
y = df['label'].values

print(f"Toplam temizlenen veri sayısı: {len(df)}")

Toplam temizlenen veri sayısı: 104928


In [12]:
X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

In [14]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=max_words, output_dim=16, input_length=max_len),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(X_train_pad, y_train, epochs=5, batch_size=32, validation_split=0.1)

loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f"\n--- Modelin Genel Başarısı ---")
print(f"Test Accuracy (Doğruluk Oranı): %{accuracy*100:.2f}")

Epoch 1/5
2361/2361 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.9431 - loss: 0.2083 - val_accuracy: 0.9402 - val_loss: 0.1930
Epoch 2/5
2361/2361 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.9484 - loss: 0.1425 - val_accuracy: 0.9504 - val_loss: 0.1257
Epoch 3/5
2361/2361 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.9588 - loss: 0.1087 - val_accuracy: 0.9607 - val_loss: 0.1110
Epoch 4/5
2361/2361 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.9649 - loss: 0.0967 - val_accuracy: 0.9618 - val_loss: 0.1057
Epoch 5/5
2361/2361 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - accuracy: 0.9681 - loss: 0.0896 - val_accuracy: 0.9645 - val_loss: 0.1025
656/656 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9683 - loss: 0.0890

--- Modelin Genel Başarısı ---
Test Accuracy (Doğruluk Oranı): %96.83


In [16]:
def yeni_yorum_test_et(metin):
    seq = tokenizer.texts_to_sequences([metin])
    pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    tahmin = model.predict(pad)[0][0]
    durum = "Pozitif" if tahmin > 0.5 else "Negatif"
    print(f"Yorum: {metin} -> Tahmin: {durum} (Olasılık: {tahmin:.2f})")

yeni_yorum_test_et("Ürün beklediğimden çok daha kaliteli çıktı, teşekkürler.")
yeni_yorum_test_et("Paranızı çöpe atmayın, paketleme berbattı.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Yorum: Ürün beklediğimden çok daha kaliteli çıktı, teşekkürler. -> Tahmin: Pozitif (Olasılık: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Yorum: Paranızı çöpe atmayın, paketleme berbattı. -> Tahmin: Pozitif (Olasılık: 0.69)
